In [4]:
import numpy as np
import os
import pandas as pd
import sys
np.random.seed(42)
random_seed = 42
import adaptive_sampling as ais
import pathlib

In [5]:
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()
resum_path = os.getenv("RESUM_PATH")
if resum_path is None:
    raise ValueError("Environment variable RESUM_PATH is not set. Make sure to define it in your .env file.")
utilities_path = os.path.join(resum_path, "utilities")
sys.path.append(utilities_path)
import utilities as utils


In [2]:
def condition(x):
    #print(x[0:3],np.sqrt(np.power(x[0],2)+np.power(x[1],2)))
    if np.sqrt(np.power(x[0],2)+np.power(x[1],2)) > 2.8:
        return 0
    if x[2] > 2. or x[2]<-4.:
        return 0
    if x[6] < 0:
        return 0
    return 1

In [3]:
version_out = 'v1.7'
version_in = 'v1.7'
nrows=None
bandwidth=0.5
n_clusters=2
with_KDE=False
n_samples_per_iter=10000
n_iterations = 1
ais_sample_iteration = 2

In [4]:
pathlib.Path(f'out/LF/{version_out}').mkdir(parents=True, exist_ok=True)
pathlib.Path(f'out/LF/{version_out}/macros').mkdir(parents=True, exist_ok=True)
pathlib.Path(f'out/LF/{version_out}/neutron-inputs-{version_out}').mkdir(parents=True, exist_ok=True)
pathlib.Path(f'out/LF/{version_out}/out').mkdir(parents=True, exist_ok=True)

trials=[0]
#for j in range(70,71):
for j in trials:
    print(f"Running {j}")
    # Open a file for writing
    n_LF = j
    
    path_to_files = f'/global/cfs/projectdirs/legend/users/aschuetz/simulation/out/low_fidelity/Neutron-Simulation-LF-{version_in}/csv/tier2/neutron-sim-LF-{version_in}-{n_LF:04}-tier2'
    data_train, sample_iter = utils.get_dataframes_concat(path_to_files, df_new=pd.DataFrame(), niterations=ais_sample_iteration, nrows=nrows)
    #sys.stdout = open(f'./out/LF/{version_out}/out/neutron-ais-output_{n_samples_per_iter}_{n_LF:04d}_{sample_iter:04d}_{version_out}.txt', "w")
    print("****************************************************************************************************")
    print(f"Starting Adaptive Importance Sampling Trial {n_LF:04d} Iteration {sample_iter} KDE {with_KDE} Refinement Iterations {n_iterations}")
    print("****************************************************************************************************")
    # Set parameter name/x_labels -> needs to be consistent with data input file
    x_labels=["x_0[m]","y_0[m]","z_0[m]","px_0[m]","py_0[m]","pz_0[m]","ekin_0[eV]"]
    y_label = 'nC_Ge77'

    x_lf = data_train[x_labels].to_numpy()
    y_lf = data_train[y_label].to_numpy()
    x_lf_sig = data_train[(data_train[y_label] >= 1)][x_labels].to_numpy()

    # Output the result
    print(f"Total rows with y(x) = 1: {x_lf_sig.shape[0]}")
    theta=data_train[["radius","thickness","npanels","theta","length"]].to_numpy()[0]
    target_distribution = ais.estimate_target_distribution(x_lf,theta, bandwidth='scott')
    
    if with_KDE==True:
        aggreated_dist=None
        proposals, kde = ais.initialize_proposals_with_kde(x_lf_sig, bandwidth=bandwidth, n_components=n_clusters)
    else: 
        data_train_all, sample_iter = utils.get_dataframes_concat(path_to_files[:-11], df_new=pd.DataFrame(), niterations=None,nrows=nrows, ending=f"-tier2_0000.csv")
        for k in range(1,ais_sample_iteration):
            data_train_all, sample_iter = utils.get_dataframes_concat(path_to_files[:-11], df_new=data_train_all, niterations=None,nrows=nrows, ending=f"-tier2_{k:04d}.csv")
        
        x_lf_sig_all = data_train_all[(data_train_all[y_label] >= 1)][x_labels].to_numpy()
        print(f"Total rows with y(x) = 1: {x_lf_sig_all.shape[0]}/ {len(data_train_all)}")
        aggreated_dist = ais.estimate_aggregated_distribution(x_lf_sig_all, bandwidth='scott')
        # Initialize Gaussian proposals from 5D data
        proposals = ais.initialize_multidimensional_proposal(x_lf_sig_all, n_clusters=n_clusters)
        print(f"Total rows with y(x) = 1: {x_lf_sig_all.shape[0]}")
        #initial_samples = kde.sample(n_samples, random_state=42)
        for proposal in proposals:
            proposal['std'] *= 1.5  # Increase standard deviation

    # Output the proposals
    print("Gaussian Proposals:")
    for i, proposal in enumerate(proposals):
        print(f"Cluster {i + 1}:")
        print(f"  Mean: {proposal['mean']}")
        print(f"  Std:  {proposal['std']}")
        print(f"  Weight: {proposal['weight']:.2f}")

    # Initialize the AdaptiveSampling class
    adaptive_sampler = ais.AdaptiveSampling(
        target_dist=target_distribution,
        initial_proposal= proposals.copy(),
        aggregated_dist=aggreated_dist,
        n_iterations=n_iterations,
        n_samples_per_iter=int(n_samples_per_iter*1.01),
        condition=condition,
        with_pca=True,
        with_KDE=with_KDE
    )
    # Run the adaptive sampling process
    samples, weights, proposals = adaptive_sampler.run(theta_k=theta)
    print("Final Proposals:")
    for i, proposal in enumerate(proposals):
        mean = ", ".join([f"{v:.2f}" for v in proposal['mean']])
        std = ", ".join([f"{v:.2f}" for v in proposal['std']])
        print(f"Component {i + 1}: Mean=[{mean}], Std=[{std}], Weight={proposal['weight']:.4f}")

    # Value to extend
    samples_new = samples[-1][0:n_samples_per_iter]
    samples_tmp= [np.append(row,0.0) for row in samples_new]
    x=np.array(samples_tmp)
    
    weights_new = weights[-1][0:n_samples_per_iter]
    weights_new /= np.sum(weights_new)  

    df_out = pd.DataFrame({
    'x[m]': ["{:.5f}".format(val) for val in x[:, 0]],
    'y[m]': ["{:.5f}".format(val) for val in x[:, 1]],
    'z[m]': ["{:.5f}".format(val) for val in x[:, 2]],
    'xmom[m]': ["{:.5f}".format(val) for val in x[:, 3]],
    'ymom[m]': ["{:.5f}".format(val) for val in x[:, 4]],
    'zmom[m]': ["{:.5f}".format(val) for val in x[:, 5]],
    'ekin[eV]': ["{:.5f}".format(val) for val in x[:, 6]],
    'time[ms]': ["{:.5f}".format(val) for val in x[:, 7]]
    })
    df_out.to_csv(f'/global/cfs/projectdirs/legend/users/aschuetz/simulation/data/neutron-inputs-{version_out}/neutron-inputs-design0_{len(x)}_{n_LF:04d}_{sample_iter:04d}_{version_out}.dat', sep = " ",header=True)
    df_out['weights']=["{:.5e}".format(val) for val in weights_new]
    df_out.to_csv(f'./out/LF/{version_out}/neutron-inputs-{version_out}/neutron-inputs2-design0_{len(x)}_{n_LF:04d}_{sample_iter:04d}_{version_out}.dat', sep = " ",header=True)
    
    #sim.print_geant4_macro(theta, n_LF, f'./out/LF/{version_out}/macros',mode='LF', version=version_out)
    
    print("Weights min:", np.min(weights_new))
    print("Weights max:", np.max(weights_new))
    print("Weights mean:", np.mean(weights_new))
    print("Weights std:", np.std(weights_new))
    #sys.stdout.close()
    #sys.stdout = sys.__stdout__
    

Running 0


100%|██████████| 2/2 [00:00<00:00, 17.67it/s]


****************************************************************************************************
Starting Adaptive Importance Sampling Trial 0000 Iteration 2 KDE False Refinement Iterations 1
****************************************************************************************************
Total rows with y(x) = 1: 16


100%|██████████| 1/1 [00:01<00:00,  1.03s/it]


Total rows with y(x) = 1: 2498/ 6010000
Total rows with y(x) = 1: 2498
Gaussian Proposals:
Cluster 1:
  Mean: [-2.39466797e-02 -1.99121516e-02 -6.83767390e-01  6.25872099e-03
 -1.61625602e-02 -1.71882527e-03  5.31564680e+05]
  Std:  [1.38482917e+00 1.43076283e+00 1.37840054e+00 8.74525961e-01
 8.66421476e-01 8.56642062e-01 4.99063012e+05]
  Weight: 0.79
Cluster 2:
  Mean: [ 5.08729343e-04 -7.20790279e-02 -6.94228390e-01 -2.50777836e-02
  2.49414343e-02 -2.79703939e-02  2.00538855e+06]
  Std:  [1.58475874e+00 1.65742148e+00 1.55174305e+00 8.37652380e-01
 9.08324449e-01 8.47767793e-01 2.05348289e+06]
  Weight: 0.21
Iteration 1/1: New proposal added.
ESS: 0.84 Entropy: 6.83
Final Proposals:
Weights min: 9.168380692712025e-06
Weights max: 0.06514289291969344
Weights mean: 9.999999999999999e-05
Weights std: 0.0011086743602827798
